In [1]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,sample_id,label,text
0,6031,0,mumia w wrote on escapenumber escapenumber esc...
1,75167,1,oh my dear sir her throat is so much better t...
2,69801,1,many more are victims of other cons telemarket...
3,78124,1,hello\nbest erection drugs\nworld wide shiping...
4,39905,0,may escapenumber escapenumber home u s world p...


In [21]:
#subtasks

def get_count(text):
    text = str(text).lower()
    count = str(text).count('free')
    return count

subtask1 = df_test.text.map(lambda x : len(x))
subtask2 = df_test.text.apply(get_count)

df_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.sample_id,
    "answer": subtask1
})

df_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.sample_id,
    "answer": subtask2
})

df_subtasks = pd.concat([df_1, df_2])

df_subtasks

,subtaskID,datapointID,answer
0,1,75707,237
1,1,4026,946
2,1,2156,1554
3,1,49593,1207
4,1,23028,1684
...,...,...,...
16685,2,41192,0
16686,2,47029,0
16687,2,24831,0
16688,2,74209,0


In [28]:
df_train.text[3]

'hello\nbest erection drugs\nworld wide shiping\nvisit us now\nhttp : / / uncapping . net / vcl / ? okok\nbroglie\n'

In [36]:
import re

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'[\d+]', '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

preprocess(df_train.text[3])

'hello best erection drugs world wide shiping visit us now http uncapping net vcl okok broglie'

In [37]:
df_train.text = df_train.text.apply(preprocess)
df_test.text = df_test.text.apply(preprocess)

df_train.head()

,sample_id,label,text
0,6031,0,mumia w wrote on escapenumber escapenumber esc...
1,75167,1,oh my dear sir her throat is so much better th...
2,69801,1,many more are victims of other cons telemarket...
3,78124,1,hello best erection drugs world wide shiping v...
4,39905,0,may escapenumber escapenumber home u s world p...


In [40]:
features = 'text'
target = 'label'

X = df_train[features]
y = df_train[target]
X_final = df_test[features]

from sklearn.feature_extraction.text import TfidfVectorizer

vect = TfidfVectorizer()
all_text = pd.concat([X, X_final])
vect.fit(all_text)
X = vect.transform(X)
X_final = vect.transform(X_final)

X

<66758x296888 sparse matrix of type '<class 'numpy.float64'>'
	with 7901498 stored elements in Compressed Sparse Row format>

In [41]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

model = LogisticRegression()

cross_val_score(model, X, y, cv=5, scoring='roc_auc')

array([0.99851277, 0.9980958 , 0.99779866, 0.99789545, 0.99799126])

In [46]:
model.fit(X, y)
y_preds = model.predict_proba(X_final)[:, 1]

In [47]:
df_preds = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": df_test.sample_id,
    "answer": y_preds
})

df_preds.head()

,subtaskID,datapointID,answer
0,3,75707,0.995670
1,3,4026,0.008904
2,3,2156,0.997476
3,3,49593,0.002669
4,3,23028,0.886524


In [48]:
submission = pd.concat([df_subtasks, df_preds])

submission

,subtaskID,datapointID,answer
0,1,75707,237.000000
1,1,4026,946.000000
2,1,2156,1554.000000
3,1,49593,1207.000000
4,1,23028,1684.000000
...,...,...,...
16685,3,41192,0.968021
16686,3,47029,0.913616
16687,3,24831,0.990382
16688,3,74209,0.979502


In [49]:
submission.to_csv("submission.csv")